# Entity Extraction Pipeline

In the previous notebook, you saw entity extraction running automatically via the memory context provider's `after_run()` hook. Here you'll look under the hood at how that extraction actually works.

Entity extraction turns unstructured conversation text into structured graph knowledge. When a user says "I'm interested in Apple's supply chain risks," the extraction pipeline identifies "Apple" as an ORGANIZATION and "supply chain risks" as a concept, then stores them as labeled nodes in Neo4j linked to the source conversation.

## What you will learn

- How the multi-stage extraction pipeline works (spaCy, GLiNER, LLM fallback)
- How to configure `MemorySettings` with extraction pipeline options
- How to manually add entities using `memory_client.long_term.add_entity()`
- How entity deduplication works with embedding similarity
- How merge strategies and resolution thresholds control what gets stored

***

Load the environment variables and import the required modules.

In [ ]:
import sys
sys.path.insert(0, '../shared')

from pydantic import SecretStr

from neo4j_agent_memory import MemoryClient, MemorySettings

from azure_embedder import get_memory_embedder
from config import Neo4jConfig

## The Extraction Pipeline

The extraction pipeline uses three stages, each with different speed/accuracy tradeoffs:

1. **spaCy** — Fast, rule-based named entity recognition. Identifies common entity types (PERSON, ORGANIZATION, LOCATION) using linguistic patterns. Runs in milliseconds but misses domain-specific entities.

2. **GLiNER** — A lightweight model specialized for entity extraction. More accurate than spaCy for uncommon entity types but requires downloading a ~500MB model from HuggingFace. Disabled in this workshop to avoid bandwidth issues.

3. **LLM fallback** — Uses the deployed Azure model to extract entities that the first two stages missed. Slowest but most flexible: it can identify any entity type you configure and understands domain context.

The pipeline runs all enabled stages and merges the results using a configurable strategy.

## Configure Extraction Settings

Configure `MemorySettings` with `extractor_type: "pipeline"` to enable the multi-stage approach. Each parameter controls a different aspect of the pipeline.

In [ ]:
neo4j_config = Neo4jConfig()

settings = MemorySettings(
    neo4j={
        "uri": neo4j_config.uri,
        "username": neo4j_config.username,
        "password": SecretStr(neo4j_config.password),
    },
    extraction={
        "extractor_type": "pipeline",
        "enable_spacy": True,
        "enable_gliner": False,  # Disabled: downloads ~500MB model, impractical in a workshop
        "enable_llm_fallback": True,
        "confidence_threshold": 0.5,  # Filter low-confidence results
        "entity_types": [
            "PERSON", "ORGANIZATION", "LOCATION", "EVENT", "OBJECT"
        ],
    },
)

print(f"Neo4j URI: {neo4j_config.uri}")
print(f"Extractor type: {settings.extraction.extractor_type}")
print(f"spaCy enabled: {settings.extraction.enable_spacy}")
print(f"GLiNER enabled: {settings.extraction.enable_gliner}")
print(f"LLM fallback enabled: {settings.extraction.enable_llm_fallback}")

## Connect the Memory Client

In [ ]:
memory_client = MemoryClient(settings, embedder=get_memory_embedder())
await memory_client.__aenter__()
print("Connected to Neo4j")

## Add a Manual Entity

While entity extraction typically runs automatically during conversations (via `extract_entities=True` on `Neo4jMicrosoftMemory`), you can also add entities manually. Entities become labeled nodes in the Neo4j knowledge graph, linked to their source conversations.

Use `memory_client.long_term.add_entity()` to add an entity with a name, type, and description. The entity gets a vector embedding and is checked for duplicates automatically.

In [ ]:
entity, dedup_result = await memory_client.long_term.add_entity(
    name="Apple Inc",
    entity_type="ORGANIZATION",
    description="Technology company that manufactures iPhone, iPad, Mac, and other consumer electronics",
)
print(f"Added entity: name={entity.name}, type={entity.entity_type}, id={entity.id}")

## Test Deduplication

Add the same entity again with slightly different wording. The deduplication system uses embedding similarity to detect that "Apple" already exists and merges the entries rather than creating a duplicate.

In [ ]:
duplicate, dedup_result = await memory_client.long_term.add_entity(
    name="Apple",
    entity_type="ORGANIZATION",
    description="Consumer electronics company known for iPhone and Mac products",
)
print(f"Result: name={duplicate.name}, id={duplicate.id}")
print(f"Same entity? {duplicate.id == entity.id}")

## Merge Strategies

When multiple pipeline stages find the same entity, the `merge_strategy` controls which result to keep:

- **`confidence`** (default) — Keeps the highest confidence score per entity. Best for quality.
- **`union`** — Keeps all unique entities from all stages. Maximum coverage.
- **`intersection`** — Only keeps entities found by multiple stages. High certainty.
- **`cascade`** — Uses first stage results, fills gaps with later stages. Fastest.

In [ ]:
settings_with_merge = MemorySettings(
    neo4j={
        "uri": neo4j_config.uri,
        "username": neo4j_config.username,
        "password": SecretStr(neo4j_config.password),
    },
    extraction={
        "extractor_type": "pipeline",
        "merge_strategy": "confidence",
    },
)
print(f"Merge strategy: {settings_with_merge.extraction.merge_strategy}")

## Resolution Settings

Entity deduplication uses embedding similarity with three configurable thresholds:

- **`exact_threshold`** (1.0) — Identical names are always merged
- **`fuzzy_threshold`** (0.85) — Similar names above this threshold are merged (e.g., "Apple" and "Apple Inc.")
- **`semantic_threshold`** (0.8) — Semantically similar entities above this threshold are merged

In [ ]:
settings_with_resolution = MemorySettings(
    neo4j={
        "uri": neo4j_config.uri,
        "username": neo4j_config.username,
        "password": SecretStr(neo4j_config.password),
    },
    resolution={
        "strategy": "composite",
        "exact_threshold": 1.0,
        "fuzzy_threshold": 0.85,
        "semantic_threshold": 0.8,
    },
)
print(f"Resolution strategy: {settings_with_resolution.resolution.strategy}")

## Cleanup

In [ ]:
await memory_client.__aexit__(None, None, None)
print("Connection closed")

## Summary

The entity extraction pipeline uses multiple stages (spaCy, GLiNER, LLM) to extract structured entities from text. Results are merged using configurable strategies and deduplicated using embedding similarity. Extracted entities become labeled nodes in Neo4j linked to their source conversations.

In this workshop, GLiNER is disabled to avoid downloading a large model over shared bandwidth. In production, enabling all three stages provides the best extraction coverage.

**What's next:** In the next notebook, you'll combine the passive context provider from Notebook 01 with **active memory tools** that give the agent explicit control over searching, saving, and recalling memory.

***

[View the complete code](../financial_data_load/solution_srcs/07_00_entity_extraction.py)

[Continue to the Memory Tools Agent Notebook](03_memory_tools_agent.ipynb)